In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, size, col, lit, when
from pyspark.sql.types import *

spark = SparkSession.builder\
        .appName("e-commerce_silver_transform")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "2g")\
        .getOrCreate()

In [ ]:
# Adjust this to your actual project root
project_root = "/home/lpascual/Projects/E-Commerce_DWH/scripts"

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from infrastructure.schemas.glue.datalake_schemas import bronze_schemas, silver_schemas

In [ ]:
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/year=2019/"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/silver"

In [ ]:
df = spark.read.parquet(data_path, schema=bronze_schemas["bronze"])
print("Number of partitions:", df.rdd.getNumPartitions())

Transformation
1) Type case columns to appropriate types
2) Break category_code to primary, secondary, tertiary
3) Split name to first_name, last_name
4) Deduplicate records


Checks
1) Confirm email is in correct format
2) Price is > 0
3) Confirm date formats
4) Replace NULL with 'UNKNOWN'

In [ ]:
#df.show(truncate=False)

In [ ]:
# Column Transformations
df_split = df.withColumn("primary_category", when(size(split(df["category_code"], "\\.")) >= 1, split(df["category_code"], "\\.").getItem(0)).otherwise(lit("unknown"))) \
.withColumn("secondary_category", when(size(split(df["category_code"], "\\.")) >= 2, split(df["category_code"], "\\.").getItem(1))) \
.withColumn("tertiary_category",  when(size(split(df["category_code"], "\\.")) >= 3, split(df["category_code"], "\\.").getItem(2))) \
.withColumn("first_name", split(df["name"], " ").getItem(0)) \
.withColumn("last_name", when(size(split(df["name"], " ")) > 2, split(df["name"], " ").getItem(2)).otherwise(split(df["name"], " ").getItem(1))) \
.withColumn("brand", when(lit(col("brand") != "NULL"), col("brand")).otherwise("unknown"))

#df_split.show(truncate=False)

In [ ]:
# Data Quality Checks Functions

def valid_email(email:str):
    return col(email).rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$")

def valid_price(price:float):
    return col(price > 0.0)

def valid_session(session:str):
    return col(session).rlike("\\w{8}-\\w{4}-\\w{4}-\\w{4}-\\w{12}")

In [ ]:
product_table = df_split.selectExpr("cast(product_id AS int) AS product_id",
                                    "product_name",
                                    "brand",
                                    "product_description",
                                    "primary_category",
                                    "secondary_category",
                                    "tertiary_category")

product_table.show(truncate=False)

In [ ]:
event_table = df_split.selectExpr("cast(event_time AS timestamp) AS event_time",
                                    "event_type",
                                    "product_id",
                                    "cast(price AS float) AS price",
                                    "cast(user_id AS int) AS user_id",
                                    "user_session")

event_table.show(truncate=False)

In [ ]:
user_table = df_split.selectExpr("cast(user_id AS int) AS user_id",
                                    "first_name",
                                    "last_name",
                                    "username",
                                    "email",
                                    "address",
                                    "city",
                                    "country",
                                    "state_code",
                                    "zip_code",
                                    "sex",)

user_table.show(truncate=False)

In [ ]:
# Write Silver Layer Tables to S3
product_table.coalesce(5).write.parquet(path=f"{output_dir}/products/", mode="overwrite")
event_table.coalesce(5).write.parquet(path=f"{output_dir}/events/", mode="overwrite")
user_table.coalesce(5).write.parquet(path=f"{output_dir}/users/", mode="overwrite")

In [ ]:
spark.stop()